# VocEd Lab 03-v2 — Stain Deconvolution & Improved Morphology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/03v2_stain_deconvolution.ipynb)

Lab 03 showed that Gaussian blur, non-local means denoising, and morphological opening/closing
produced only marginal improvements over Lab 02.  Why?  Because they all wrapped the same
grayscale threshold — and if nucleus and cytoplasm overlap in brightness, no amount of
pre-processing can fix that.

This lab attacks the problem at the *representation* level.  H&E stained slides encode cell
identity in **colour**: haematoxylin stains nuclei dark purple; eosin stains cytoplasm pink.
Converting to grayscale throws that information away entirely.

**Stain deconvolution** (Ruifrok & Johnston, 2001) inverts the Beer–Lambert absorption model
to recover the concentration of each stain at every pixel.  The result is two independent
scalar maps — one tuned to nuclei, one to cytoplasm — that are far easier to threshold than
grayscale brightness.

**New pipeline:**  
`RGB → stain deconvolution → per-stain thresholds → improved morphology → Bayesian optimisation`

By the end of this lab you will be able to:
- Explain why grayscale is a poor representation for H&E cytology images.
- Apply stain deconvolution to separate haematoxylin and eosin channels.
- Build a two-channel threshold pipeline and evaluate it with the Dice score.
- Use hole-filling and small-object removal as targeted morphological post-processing.
- Re-run Bayesian optimisation on the new pipeline and compare results.
- Compute N/C ratio predictions and the Pearson correlation coefficient.

## 0. Setup

In [ ]:
!pip install scikit-optimize scikit-image --quiet

# Clone the repo (skip if already present)
!git clone https://github.com/emilsar/VocEd.git 2>/dev/null || echo 'Already cloned.'
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import skimage.color as skc
import skimage.morphology as skm
from scipy.ndimage import binary_fill_holes
from skopt import gp_minimize
from skopt.space import Real

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Same split as every previous lab — same seed, same indices
np.random.seed(42)
idx       = np.random.permutation(N)
train_idx = idx[:160]
test_idx  = idx[160:]

def to_gray(img):
    """Standard ITU-R grayscale from a (3, H, W) float32 image."""
    return 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]

def segment_gray(img, t_nuc, t_bg):
    """Lab 01/02 grayscale threshold pipeline — kept here for comparison."""
    gray = to_gray(img)
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nuc]                           = 2
    pred[(gray >= t_nuc) & (gray < t_bg)]        = 1
    return pred

def dice_score(pred, target, cls):
    p = pred   == cls
    t = target == cls
    inter = (p & t).sum()
    denom = p.sum() + t.sum()
    return 1.0 if denom == 0 else 2 * inter / denom

print(f'Loaded {N} images.  Train: {len(train_idx)}  Test: {len(test_idx)}')

## 2. Stain Deconvolution

H&E staining uses two dyes with distinct spectral signatures:

| Stain | Target | Colour |
|---|---|---|
| **Haematoxylin** | Nucleus (DNA) | Dark purple/blue |
| **Eosin** | Cytoplasm (proteins) | Pink/red |

Standard grayscale conversion collapses all three colour channels into a single brightness
value.  A dark nucleus and a mid-tone cytoplasm can end up at overlapping grayscale values,
making any threshold imprecise.

Stain deconvolution inverts the Beer–Lambert absorption model:

$$\text{optical density} = -\log(I / I_0)$$

where $I$ is the transmitted light intensity and $I_0$ is the background.  Each stain
absorbs light at a known spectral profile.  Inverting the mixing matrix gives per-stain
concentration maps.  `scikit-image` ships the haematoxylin–eosin–DAB (HED) matrix, so
one function call does the work:

```python
stains = skc.separate_stains(img_hwc, skc.hed_from_rgb)
haematoxylin = stains[:, :, 0]   # high values → nucleus
eosin        = stains[:, :, 1]   # high values → cytoplasm
```

Notice in the visualisation below how much cleaner the class separation is compared to grayscale.

In [ ]:
IDX = 7
img7     = X[IDX].transpose(1, 2, 0)              # (256, 256, 3) — skimage is channel-last

stains7  = skc.separate_stains(img7, skc.hed_from_rgb)
hem7     = stains7[:, :, 0]    # haematoxylin — nucleus signal
eos7     = stains7[:, :, 1]    # eosin        — cytoplasm signal
gray7    = to_gray(X[IDX])     # grayscale for comparison

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img7);                    axes[0].set_title('Original RGB');            axes[0].axis('off')
axes[1].imshow(gray7,  cmap='gray');     axes[1].set_title('Grayscale (Labs 01–03)');  axes[1].axis('off')
axes[2].imshow(hem7,   cmap='Purples');  axes[2].set_title('Haematoxylin (nucleus)');  axes[2].axis('off')
axes[3].imshow(eos7,   cmap='RdPu');     axes[3].set_title('Eosin (cytoplasm)');       axes[3].axis('off')
plt.tight_layout()
plt.show()

# Print mean channel values inside each ground-truth class
mask7 = y[IDX]
print('Mean channel values by ground-truth class:')
print(f'  Grayscale:      nucleus={gray7[mask7==2].mean():.3f}   cytoplasm={gray7[mask7==1].mean():.3f}   background={gray7[mask7==0].mean():.3f}')
print(f'  Haematoxylin:   nucleus={hem7[mask7==2].mean():.3f}   cytoplasm={hem7[mask7==1].mean():.3f}   background={hem7[mask7==0].mean():.3f}')
print(f'  Eosin:          nucleus={eos7[mask7==2].mean():.3f}   cytoplasm={eos7[mask7==1].mean():.3f}   background={eos7[mask7==0].mean():.3f}')
print()
print('Larger gap between nucleus and cytoplasm means the threshold has more room to land correctly.')

## 3. The Stain Deconvolution Pipeline

The new pipeline has three stages:

1. **Stain deconvolution** — decompose RGB into independent haematoxylin and eosin maps.
2. **Per-stain thresholding** — `haematoxylin > t_hem` → nucleus; `eosin > t_eos` AND NOT nucleus → cytoplasm; everything else → background.
3. **Improved morphology** — instead of generic opening/closing on a disk, we use three targeted operations:
   - `binary_fill_holes` — plugs interior holes in the nucleus mask (common when staining is uneven across the nucleus body).
   - `remove_small_objects` — removes isolated specks smaller than `min_size` pixels (staining artefacts, debris).
   - `closing` — smooths the boundary of each mask without discarding entire regions.

The key insight: instead of *one* grayscale threshold trying to resolve three classes at once,
we now have *two independent signals* — one tuned physically to nuclei, one to cytoplasm.

In [ ]:
def stain_pipeline(img, t_hem, t_eos, use_morph=True, morph_radius=2):
    """
    Stain deconvolution segmentation pipeline.

    img          : (3, H, W) float32 image, channels-first
    t_hem        : haematoxylin threshold — pixels above this become nucleus (class 2)
    t_eos        : eosin threshold        — pixels above this become cytoplasm (class 1)
    use_morph    : apply improved morphological post-processing
    morph_radius : disk radius for closing step (smooths mask boundaries)
    """
    img_hwc = img.transpose(1, 2, 0)                   # (H, W, 3) for skimage

    # ── Step 1: stain deconvolution ───────────────────────────────────────────
    stains   = skc.separate_stains(img_hwc, skc.hed_from_rgb)
    hem      = stains[:, :, 0]                          # haematoxylin concentration
    eos      = stains[:, :, 1]                          # eosin concentration

    # ── Step 2: per-stain thresholding ────────────────────────────────────────
    # Nucleus: strong haematoxylin signal
    nuc_mask  = hem > t_hem
    # Cytoplasm: strong eosin signal, but NOT where nucleus already claimed the pixel
    cyto_mask = (eos > t_eos) & ~nuc_mask

    # ── Step 3: improved morphology ───────────────────────────────────────────
    if use_morph:
        disk = skm.disk(morph_radius)

        # Nucleus: fill interior holes → remove small debris → smooth boundary
        nuc_mask  = binary_fill_holes(nuc_mask)
        nuc_mask  = skm.remove_small_objects(nuc_mask, min_size=50)
        nuc_mask  = skm.closing(nuc_mask, disk)

        # Cytoplasm: fill holes → remove small artefacts
        cyto_mask = binary_fill_holes(cyto_mask)
        cyto_mask = skm.remove_small_objects(cyto_mask, min_size=50)
        # Re-exclude nucleus region in case closing expanded into it
        cyto_mask = cyto_mask & ~nuc_mask

    # ── Combine: nucleus takes priority over cytoplasm ────────────────────────
    pred = np.zeros(hem.shape, dtype=np.int64)
    pred[cyto_mask] = 1
    pred[nuc_mask]  = 2

    return pred


# Sanity check
pred7 = stain_pipeline(X[7], t_hem=0.15, t_eos=0.05)
print('Prediction classes present:', np.unique(pred7))
print('Prediction shape:', pred7.shape)

In [ ]:
# Visual comparison: Lab 03 grayscale vs Lab 03-v2 stain pipeline on image 7
# Mask colours (tab10 built-in): class 0 background=blue  class 1 cytoplasm=orange  class 2 nucleus=green

pred_gray7  = segment_gray(X[7], 0.45, 0.85)
pred_stain7 = stain_pipeline(X[7], t_hem=0.15, t_eos=0.05)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(X[7].transpose(1, 2, 0))
axes[0].set_title('Original RGB');  axes[0].axis('off')

axes[1].imshow(y[7], cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[1].set_title('Ground truth');  axes[1].axis('off')

axes[2].imshow(pred_gray7, cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[2].set_title('Lab 03 — grayscale threshold');  axes[2].axis('off')

axes[3].imshow(pred_stain7, cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[3].set_title('Lab 03-v2 — stain deconvolution');  axes[3].axis('off')

plt.tight_layout()
plt.show()

# Print per-class Dice for this one image
for method, pred in [('grayscale', pred_gray7), ('stain deconv', pred_stain7)]:
    d1 = dice_score(pred, y[7], 1)
    d2 = dice_score(pred, y[7], 2)
    print(f'Image 7 — {method:15s}  cytoplasm Dice={d1:.3f}  nucleus Dice={d2:.3f}  mean={0.5*(d1+d2):.3f}')

## 4. Bayesian Optimisation

We now search for the best combination of three parameters:

| Parameter | Meaning | Search range |
|---|---|---|
| `t_hem` | Haematoxylin threshold — controls how much of the image becomes nucleus | [0.01, 0.60] |
| `t_eos` | Eosin threshold — controls cytoplasm sensitivity | [0.01, 0.40] |
| `morph_radius` | Disk radius for the closing step | [1, 5] |

The objective is identical to Labs 02 and 03: maximise mean Dice over the training set.
This allows a fair comparison — same budget (50 evaluations), same train/test split.

> **Note:** each evaluation runs `stain_pipeline` over all 160 training images.  Expect
> this cell to take **3–6 minutes** depending on hardware.

In [ ]:
def objective_stain(params):
    t_hem, t_eos, radius = params
    scores = []
    for i in train_idx:
        pred = stain_pipeline(X[i], t_hem, t_eos,
                              use_morph=True, morph_radius=int(radius))
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

search_space = [
    Real(0.01, 0.60, name='t_hem'),
    Real(0.01, 0.40, name='t_eos'),
    Real(1,    5,    name='morph_radius'),
]

print('Running Bayesian optimisation (50 evals)…')
result = gp_minimize(
    func=objective_stain,
    dimensions=search_space,
    n_calls=50,
    n_initial_points=10,
    random_state=42,
    verbose=False,
)
print('Done.')

best_hem, best_eos, best_radius = result.x
print(f'Best params:  t_hem={best_hem:.4f}   t_eos={best_eos:.4f}   morph_radius={best_radius:.1f}')
print(f'Train Dice:   {-result.fun:.4f}')

## 5. Dice Comparison — Labs 01, 02, 03-v2

We evaluate every method on the held-out test set (40 images).  Lab 02's Bayesian
optimisation is re-run with the same budget to provide a fair grayscale baseline.

In [ ]:
# ── Re-run Lab 02 optimisation for a fair grayscale baseline ──────────────────
def objective_lab02(params):
    t_nuc, t_bg = params
    scores = []
    for i in train_idx:
        pred = segment_gray(X[i], t_nuc, t_bg)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

print('Re-running Lab 02 Bayesian optimisation…')
res02 = gp_minimize(objective_lab02,
                    [Real(0.10, 0.70), Real(0.50, 0.99)],
                    n_calls=50, n_initial_points=10, random_state=42, verbose=False)
print('Done.')

# ── Evaluate all methods on the test set ─────────────────────────────────────
def test_dice_mean(pred_fn):
    scores = []
    for i in test_idx:
        pred = pred_fn(i)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return np.mean(scores)

d_lab01   = test_dice_mean(lambda i: segment_gray(X[i], 0.45, 0.85))
d_lab02   = test_dice_mean(lambda i: segment_gray(X[i], res02.x[0], res02.x[1]))
d_lab03v2 = test_dice_mean(
    lambda i: stain_pipeline(X[i], best_hem, best_eos,
                             use_morph=True, morph_radius=int(best_radius))
)

print()
print('=' * 58)
print(f'{"Method":<40}  {"Test Dice":>10}')
print('-' * 58)
print(f'{"Lab 01 — hand-picked grayscale thresholds":<40}  {d_lab01:10.4f}')
print(f'{"Lab 02 — Bayesian opt (grayscale)":<40}  {d_lab02:10.4f}')
print(f'{"Lab 03-v2 — stain deconv + morphology":<40}  {d_lab03v2:10.4f}')
print('=' * 58)

## 6. N/C Ratio Prediction & Pearson Correlation Coefficient

A high Dice score tells us the *shape* of the predicted mask is close to ground truth.
But the clinical question is different: how accurately does our model estimate the
*nucleus-to-cytoplasm (N/C) ratio*?

$$\text{N/C ratio} = \frac{\text{nucleus pixels}}{\text{cytoplasm pixels}}$$

A high N/C ratio is a hallmark of malignancy.  We evaluate two things:

1. **Scatter plot** — predicted vs ground-truth ratio.  Points on the diagonal = perfect prediction.
2. **Pearson correlation coefficient** *r* — how linearly correlated are the predictions?
   A value of *r* = 1.0 is a perfect linear relationship; *r* = 0 means no correlation.

$$r = \frac{\sum_i (p_i - \bar{p})(g_i - \bar{g})}{\sqrt{\sum_i (p_i - \bar{p})^2 \sum_i (g_i - \bar{g})^2}}$$

`np.corrcoef` computes this directly — no extra imports needed.

In [ ]:
def nc_ratio(mask):
    """N/C ratio from a (H, W) integer mask.  Returns NaN if cytoplasm is absent."""
    nuc  = (mask == 2).sum()
    cyto = (mask == 1).sum()
    return float('nan') if cyto == 0 else nuc / cyto

# Ground-truth N/C ratios for the test set
gt_ratios = np.array([nc_ratio(y[i]) for i in test_idx])
valid     = ~np.isnan(gt_ratios)
gt        = gt_ratios[valid]
idx_v     = test_idx[valid]

print(f'Valid test images: {valid.sum()} / {len(test_idx)}')
print(f'Ground-truth N/C — mean: {gt.mean():.3f}  std: {gt.std():.3f}  range: [{gt.min():.3f}, {gt.max():.3f}]')

# Predicted N/C ratios from each method
pred_lab01   = np.array([nc_ratio(segment_gray(X[i], 0.45, 0.85))                         for i in idx_v])
pred_lab02   = np.array([nc_ratio(segment_gray(X[i], res02.x[0], res02.x[1]))             for i in idx_v])
pred_lab03v2 = np.array([nc_ratio(stain_pipeline(X[i], best_hem, best_eos,
                                                  use_morph=True, morph_radius=int(best_radius)))
                          for i in idx_v])

# Pearson r — computed using np.corrcoef; NaN predictions are excluded
def pearson_r(pred, gt_vals):
    ok = ~np.isnan(pred)
    return np.corrcoef(pred[ok], gt_vals[ok])[0, 1]

r_lab01   = pearson_r(pred_lab01,   gt)
r_lab02   = pearson_r(pred_lab02,   gt)
r_lab03v2 = pearson_r(pred_lab03v2, gt)

print()
print(f'Pearson r — Lab 01: {r_lab01:.4f}   Lab 02: {r_lab02:.4f}   Lab 03-v2: {r_lab03v2:.4f}')

In [ ]:
lim = max(gt.max(),
          np.nanmax(pred_lab01),
          np.nanmax(pred_lab02),
          np.nanmax(pred_lab03v2)) * 1.1

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (title, pred, r, color) in zip(axes, [
    ('Lab 01 — hand-picked',     pred_lab01,   r_lab01,   'steelblue'),
    ('Lab 02 — Bayesian',        pred_lab02,   r_lab02,   'darkorange'),
    ('Lab 03-v2 — stain deconv', pred_lab03v2, r_lab03v2, 'seagreen'),
]):
    ok = ~np.isnan(pred)
    ax.scatter(gt[ok], pred[ok], alpha=0.6, color=color, s=30)
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
    ax.set_xlabel('Ground-truth N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{title}\nr = {r:.4f}')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print('Points near the dashed diagonal = accurate predictions.')
print('Points scattered far off the line = systematic over- or under-estimation of the N/C ratio.')

## Wrap-up

**Key takeaways:**

- **Representation matters more than polish.**  Labs 01–03 all applied increasingly sophisticated
  preprocessing to the same grayscale signal.  Switching to stain deconvolution — a change that
  costs one function call — provides a physically grounded signal that no amount of blurring or
  morphology could simulate.

- **Targeted morphology outperforms generic operations.**  Hole-filling and small-object removal
  address specific failure modes (uneven staining inside nuclei, debris artefacts) rather than
  blindly eroding or dilating everything.

- **Two metrics tell different stories.**  Dice score measures how well the mask *shape* matches.
  Pearson *r* measures how well the *ratio* generalises across images.  A method can have decent
  Dice but poor *r* if it has a systematic bias (always over-estimating nucleus size, for example).

- **The grayscale ceiling is real.**  If the Dice improvement here is substantial, it confirms
  that the bottleneck in Labs 01–03 was the representation, not the threshold logic or the
  morphological steps.

In the next lab we abandon thresholds entirely and treat each pixel as a point in 3-D colour
space, training a k-NN classifier to separate the classes directly.

---

## Group Exercise — Ablation Study

Each person tests one component of the new pipeline on `test_idx[0:5]`, using the
optimised thresholds (`best_hem`, `best_eos`).

| Person | Task |
|---|---|
| A | Run `stain_pipeline` with `use_morph=False`.  How much does morphology contribute to Dice? |
| B | Replace stain deconvolution with grayscale: use `segment_gray` with `res02.x[0]`, `res02.x[1]`.  What is the Dice gap vs `stain_pipeline`? |
| C | Print `hem7.min()`, `hem7.max()`, `eos7.min()`, `eos7.max()`.  What does the channel range tell you about how to choose good threshold values? |

Share your numbers and discuss: which component — stain deconvolution or improved morphology
— accounts for more of the gain over Lab 02?